In [12]:
#these are the liberaies I used for the whole project, but for preprocessing not all of them are used
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np
import os

from pathlib import Path
#from aerosound.params import *
from keras import Sequential, layers
from keras.metrics import Recall
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
import inspect

## Convert .wav to .csv

In [14]:
# change to the correct folder in order to be able to import the following package
print(os.getcwd())


/Users/shaotongwang/code/Marie-Jourdain/aerosound


In [3]:
os.chdir("../..")
print(os.getcwd())

/Users/shaotongwang/code/Marie-Jourdain/aerosound


In [20]:
from aerosound.ml_logic.data import *
from aerosound.ml_logic.preprocessing import *
# I updated the preprocessing.py, so please use the pull the preprocessing.py in my branch

In [ ]:
def get_path (path):
    #input the path to the classification .csv  (fan only)
    df_path = pd.read_csv(path)
    df_path = df_path[df_path['machine'] == 'fan']
    root = Path("raw_data")
    audio_paths = [
        root / row.noise / row.id / row.label / row.file
        for row in df_path.itertuples(index=False)
    ]
    return audio_paths


In [46]:
def get_data_rnn(audio_paths):
# this function is used to convert many .wav files in folder with subfolders to a dictionary of dataframes
    # 1, obtain data linked with the given path,
    # 2, convert it from .wav to dataframe,
    # 3, saperated the data by machine type (key) and save the dataframes (value) into a dictionary,
    # 4, return to this dictionary
    # audio_paths = list(Path(path).rglob("*.wav"))
    df = build_dataset(audio_paths)
    # machine_dfs = {}

    # for machine in df['machine'].unique():
    #     machine_dfs[machine] = df[
    #         df['machine'] == machine
    #     ].copy()

    #print(machine_dfs.keys())
    return df

In [45]:
def get_dataframe(machine_dfs, machine_name):
    #obtain the dataframe of the certain machin
    return machine_dfs[machine_name]

In [39]:
def get_data_single_wav(path):
# this is the function that used for API to convert one .wav to dataframe
    audio_paths = list(Path(path).rglob("*.wav"))
    df = build_dataset(audio_paths)
    return df

In [40]:
path_for_train = 'aerosound/model/df_train.csv'
path_for_val = 'aerosound/model/df_validate.csv'
path_for_test = 'aerosound/model/df_test.csv'

In [41]:
path_list_train = get_path(path_for_train)
path_list_val = get_path(path_for_val)
path_list_test = get_path(path_for_test)
path_list_train

[PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000000.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000006.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000010.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000016.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000020.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000025.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000030.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000035.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000040.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000045.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000050.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000056.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000061.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000065.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000070.wav'),
 PosixPath('raw_data/-6_dB_fan/id_00/abnormal/00000075.wav'),
 PosixPa

In [47]:
df_fan_train = get_data_rnn(path_list_train)
df_fan_val = get_data_rnn(path_list_val)
df_fan_test = get_data_rnn(path_list_test)
df_fan_train

,noise,machine,id,label,file,snr,f_000,f_001,f_002,f_003,...,f_190,f_191,f_192,f_193,f_194,f_195,f_196,f_197,f_198,f_199
0,-6_dB,dB_fan,id_00,abnormal,00000000.wav,-6,0.011441,0.080050,0.009736,0.095685,...,0.010477,0.082552,0.011197,0.076923,0.012078,0.064415,0.012113,0.069418,0.010662,0.088180
1,-6_dB,dB_fan,id_00,abnormal,00000006.wav,-6,0.011929,0.061288,0.014182,0.040025,...,0.010839,0.059412,0.013038,0.050657,0.015023,0.050031,0.012074,0.063164,0.013519,0.041901
2,-6_dB,dB_fan,id_00,abnormal,00000010.wav,-6,0.011440,0.061914,0.011594,0.065666,...,0.010713,0.088180,0.011881,0.081301,0.012925,0.083177,0.013544,0.089431,0.013099,0.081926
3,-6_dB,dB_fan,id_00,abnormal,00000016.wav,-6,0.013235,0.025016,0.012820,0.025641,...,0.010213,0.026266,0.010595,0.033146,0.011691,0.017511,0.011270,0.028768,0.011264,0.025641
4,-6_dB,dB_fan,id_00,abnormal,00000020.wav,-6,0.011455,0.066917,0.012531,0.066917,...,0.011697,0.069418,0.010888,0.080675,0.012445,0.086929,0.011725,0.080050,0.012543,0.083802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,6_dB,dB_fan,id_06,normal,00000992.wav,6,0.005764,0.102564,0.005547,0.096936,...,0.005664,0.108193,0.006197,0.093809,0.006027,0.121951,0.005186,0.105691,0.006067,0.106942
9986,6_dB,dB_fan,id_06,normal,00000997.wav,6,0.005981,0.098812,0.005665,0.087555,...,0.006540,0.081926,0.006005,0.077548,0.005486,0.073171,0.006283,0.084428,0.006277,0.093183
9987,6_dB,dB_fan,id_06,normal,00001002.wav,6,0.005544,0.093183,0.005326,0.098186,...,0.005645,0.077548,0.005052,0.087555,0.004966,0.090682,0.004986,0.093183,0.005110,0.088180
9988,6_dB,dB_fan,id_06,normal,00001008.wav,6,0.005867,0.074422,0.007253,0.071920,...,0.006802,0.060663,0.006826,0.063164,0.006259,0.065666,0.006959,0.059412,0.006802,0.061288


In [48]:
df_fan_train.to_csv("aerosound/model/fan_all_train_stdzcr_100.csv", index=False)

In [49]:
df_fan_val.to_csv("aerosound/model/fan_all_val_stdzcr_100.csv", index=False)

In [50]:
df_fan_test.to_csv("aerosound/model/fan_all_test_stdzcr_100.csv", index=False)

In [ ]:
import aerosound.ml_logic.preprocessing as prep
import aerosound.ml_logic.data as da

#print(inspect.getsource(prep.extract_type_features))
#print(inspect.getsource(prep.preprocess_audio))
print(inspect.getsource(da.build_dataset))

def build_dataset(audio_paths):

    metadata_df = build_metadata_df(audio_paths)

    features = build_features(audio_paths)

    # (5550, 100, 2) -> (5550, 200)
    features = features.reshape(features.shape[0], -1)

    # Generate 200 column names
    feature_cols = [
        f"f_{i:03d}" for i in range(features.shape[1])
    ]

    features_df = pd.DataFrame(
        features,
        columns=feature_cols
    )

    metadata_features_df = pd.concat(
        [metadata_df, features_df],
        axis=1
    )

    return metadata_features_df



In [ ]:
machine_dfs = get_data_rnn('raw_data/0_dB_fan')
data_ori = get_dataframe(machine_dfs, 'fan')
# then you get the dataframe of the fan:
data_ori

dict_keys(['fan'])


,noise,machine,id,label,file,snr,f_000,f_001,f_002,f_003,...,f_190,f_191,f_192,f_193,f_194,f_195,f_196,f_197,f_198,f_199
0,0_dB,fan,id_00,abnormal,00000059.wav,0,0.008489,0.047530,0.009500,0.043777,...,0.009530,0.045654,0.010291,0.039400,0.009986,0.028143,0.009617,0.049406,0.009319,0.044403
1,0_dB,fan,id_00,abnormal,00000071.wav,0,0.008590,0.036898,0.009991,0.026892,...,0.010346,0.026892,0.008103,0.031895,0.009203,0.033146,0.008105,0.034396,0.007748,0.040025
2,0_dB,fan,id_00,abnormal,00000065.wav,0,0.010086,0.038149,0.009601,0.053158,...,0.010024,0.045654,0.008293,0.050657,0.008500,0.038149,0.008613,0.037523,0.008502,0.053158
3,0_dB,fan,id_00,abnormal,00000273.wav,0,0.006436,0.055660,0.007534,0.071920,...,0.004920,0.095685,0.005144,0.082552,0.005650,0.083177,0.005638,0.078799,0.005438,0.085053
4,0_dB,fan,id_00,abnormal,00000267.wav,0,0.004283,0.051282,0.003697,0.053784,...,0.006067,0.249531,0.005918,0.207004,0.005611,0.191370,0.005764,0.148218,0.004165,0.158849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5545,0_dB,fan,id_02,normal,00000068.wav,0,0.007421,0.103189,0.007069,0.107567,...,0.006890,0.114447,0.006674,0.101939,0.007352,0.095685,0.007334,0.101939,0.006161,0.119450
5546,0_dB,fan,id_02,normal,00000040.wav,0,0.005930,0.126329,0.006486,0.108818,...,0.007167,0.093183,0.006789,0.101313,0.006511,0.113196,0.006546,0.115072,0.006452,0.115697
5547,0_dB,fan,id_02,normal,00000726.wav,0,0.006656,0.118824,0.006478,0.110694,...,0.006791,0.093183,0.005945,0.132583,0.006180,0.124453,0.005910,0.118199,0.006244,0.109443
5548,0_dB,fan,id_02,normal,00000732.wav,0,0.006282,0.112570,0.006609,0.108818,...,0.006236,0.100688,0.006337,0.092558,0.006249,0.103189,0.005911,0.106942,0.006746,0.099437


In [10]:
# save the dataframe to .csv
data_ori.to_csv("aerosound/model/0db_fan_stdzcr_100.csv", index=False)